<a href="https://colab.research.google.com/github/Tasan99/Assignments/blob/main/SD4_Synthetic_Loan_Data_Generation_with_CTGAN_%26_TVAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we will evaluate two techniques to generate structured synthetic data: **Tabular GAN** and **Tabular Variational Autoencoder**.

#Part 1: Tabular GANs
Tabular GANs are a type of Generative Adversarial Network (GAN) specifically designed to generate synthetic tabular data (data organized in rows and columns, like a spreadsheet or a Pandas DataFrame) that closely resembles a real-world dataset. Traditional GANs were initially more successful in generating continuous data like images. Tabular data presents unique challenges due to the presence of:
* Mixed Data Types: Tables often contain both numerical (continuous or discrete) and categorical features.
* Complex Correlations: Features in a table can have intricate linear and non-linear relationships.
* Unbalanced Categories: Categorical features can have classes with highly varying frequencies.
* Discrete Values: Even numerical columns might represent discrete quantities.


CTGAN (Conditional Tabular Generative Adversarial Network) addresses these challenges through several key innovations built upon the standard GAN architecture:
* Generator (G):
Takes random noise as input.
Its goal is to generate synthetic data samples that the discriminator cannot distinguish from real data.
It uses neural networks (typically Multi-Layer Perceptrons or MLPs) to transform the noise into synthetic tabular data.
* Discriminator (D):
Takes a batch of data as input, which can be a mix of real data samples from the original dataset and synthetic data samples generated by the generator.
Its goal is to correctly classify each input sample as either "real" or "synthetic."
It also uses neural networks (MLPs) for this classification task.
* Adversarial Training:
The generator and discriminator are trained in an adversarial manner.
The generator tries to fool the discriminator by producing increasingly realistic synthetic data.
The discriminator tries to become better at distinguishing real from synthetic data.
This competition drives both networks to improve, ideally leading the generator to produce synthetic data that is statistically very similar to the real data.


In essence, CTGAN aims to learn the underlying data generation process of  tabular dataset by training a generator to produce synthetic data that fools a discriminator trained to distinguish it from the real data.

In [3]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors


## 1. Load and inspect `AnguloM/loan_data`

The dataset is a LendingClub‑style consumer loan dataset hosted on Hugging Face.  
Key fields (from the dataset card):

- `not.fully.paid`: **outcome** – 1 if the loan was *not* fully repaid (default/charge‑off), 0 otherwise. [web:111]
- `credit.policy`: 1 if the customer meets LendingClub's underwriting criteria.
- `purpose`: loan purpose (debt_consolidation, credit_card, etc.).
- Numeric features: `int.rate`, `installment`, `log.annual.inc`, `dti`, `fico`, `days.with.cr.line`, `revol.bal`, `revol.util`, `inq.last.6mths`, `delinq.2yrs`, `pub.rec`. [web:109][web:142]

We will:
1. Load the dataset.
2. Inspect schema and basic statistics.
3. Confirm class balance of `not.fully.paid`.


In [5]:
loan_ds = load_dataset("AnguloM/loan_data")
df = loan_ds["train"].to_pandas()

df.head()

README.md: 0.00B [00:00, ?B/s]

loan.zip:   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9578 [00:00<?, ? examples/s]

,credit.policy,purpose,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9578 entries, 0 to 9577
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   credit.policy      9578 non-null   int64  
 1   purpose            9578 non-null   object 
 2   int.rate           9578 non-null   float64
 3   installment        9578 non-null   float64
 4   log.annual.inc     9578 non-null   float64
 5   dti                9578 non-null   float64
 6   fico               9578 non-null   int64  
 7   days.with.cr.line  9578 non-null   float64
 8   revol.bal          9578 non-null   int64  
 9   revol.util         9578 non-null   float64
 10  inq.last.6mths     9578 non-null   int64  
 11  delinq.2yrs        9578 non-null   int64  
 12  pub.rec            9578 non-null   int64  
 13  not.fully.paid     9578 non-null   int64  
dtypes: float64(6), int64(7), object(1)
memory usage: 1.0+ MB


In [7]:
df["not.fully.paid"].value_counts(normalize=True)


,proportion
not.fully.paid,
0,0.839946
1,0.160054


## 2. Preprocessing with `not.fully.paid` as outcome

Our prediction / label variable is:

- `not.fully.paid` (1 = default / not fully repaid, 0 = fully paid).

We define three groups of columns:

- **Target:** `not.fully.paid`
- **Categorical features:** `purpose`, `credit.policy` (treated as discrete category).
- **Numeric features:** rate, installment, income, FICO, etc.

Steps:
1. Drop rows with missing values (dataset is usually clean, but we are defensive).
2. Split into features `X` and target `y`.


In [8]:
# Drop any NA rows to simplify the lab
df = df.dropna().reset_index(drop=True)

target_col = "not.fully.paid"

cat_cols = ["purpose", "credit.policy"]
num_cols = [
    "int.rate",
    "installment",
    "log.annual.inc",
    "dti",
    "fico",
    "days.with.cr.line",
    "revol.bal",
    "revol.util",
    "inq.last.6mths",
    "delinq.2yrs",
    "pub.rec"
]

X = df[cat_cols + num_cols].copy()
y = df[target_col].astype(int)

df[[target_col]].value_counts(normalize=True)


,proportion
not.fully.paid,
0,0.839946
1,0.160054


## 3. Train/test split on real data

We will later:

- Train a classifier on **real data** (baseline).
- Train a classifier on **synthetic data** and test on **real data** (TSTR).

To do this, we split the real dataset into `train` and `test` with stratification on `not.fully.paid`.


In [9]:
real_train, real_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

real_train.shape, real_test.shape


((7662, 14), (1916, 14))

## 4. Train CTGAN to generate synthetic loans

We will use **CTGAN**, a GAN variant designed specifically for mixed‑type tabular data.

Key design choices:

- Input to CTGAN: all feature columns **plus** the outcome `not.fully.paid`.
- `discrete_columns` parameter: includes all categorical fields and integer count features, including the binary outcome.

This lets us later **condition** on `not.fully.paid` if we want to oversample defaulted loans.


In [10]:
from ctgan import CTGAN

ctgan_data = df[cat_cols + num_cols + [target_col]].copy()

discrete_cols = cat_cols + ["inq.last.6mths", "delinq.2yrs", "pub.rec", target_col]

ctgan = CTGAN(
    epochs=300,       # increase for better quality if you have time/compute
    batch_size=500,
    verbose=True
)

ctgan.fit(ctgan_data, discrete_columns=discrete_cols)

Gen. (-00.95) | Discrim. (-00.28): 100%|██████████| 300/300 [03:01<00:00,  1.66it/s]


## 5. Generate synthetic data

Let's generate 10,000 synthetic loan records.

We will inspect:

- Schema (column names, dtypes).
- Class balance for `not.fully.paid` in synthetic data vs. real data.


In [11]:
n_synth = 10_000
synthetic = ctgan.sample(n_synth)
synthetic.head()


,purpose,credit.policy,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,major_purchase,1,0.119752,332.641201,10.959364,13.952289,757,3306.892127,-817,-2.371223,1,0,0,0
1,all_other,1,0.080889,220.977660,9.624589,6.505947,742,6777.388473,-797,-1.058843,3,1,3,0
2,all_other,1,0.140480,255.551495,11.063740,5.565733,782,4032.651752,956,35.813879,1,0,0,1
3,educational,0,0.145875,111.511720,10.375560,4.064979,677,815.173103,4788,51.135042,19,0,0,1
4,small_business,1,0.124912,156.583024,11.889305,19.390320,651,4529.535532,7302,36.615176,0,2,0,0


In [12]:
synthetic.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   purpose            10000 non-null  object 
 1   credit.policy      10000 non-null  int64  
 2   int.rate           10000 non-null  float64
 3   installment        10000 non-null  float64
 4   log.annual.inc     10000 non-null  float64
 5   dti                10000 non-null  float64
 6   fico               10000 non-null  int64  
 7   days.with.cr.line  10000 non-null  float64
 8   revol.bal          10000 non-null  int64  
 9   revol.util         10000 non-null  float64
 10  inq.last.6mths     10000 non-null  int64  
 11  delinq.2yrs        10000 non-null  int64  
 12  pub.rec            10000 non-null  int64  
 13  not.fully.paid     10000 non-null  int64  
dtypes: float64(6), int64(7), object(1)
memory usage: 1.1+ MB


In [13]:
print("Real outcome distribution:")
print(df[target_col].value_counts(normalize=True))

print("\nSynthetic outcome distribution:")
print(synthetic[target_col].value_counts(normalize=True))


Real outcome distribution:
not.fully.paid
0    0.839946
1    0.160054
Name: proportion, dtype: float64

Synthetic outcome distribution:
not.fully.paid
0    0.8039
1    0.1961
Name: proportion, dtype: float64


### 5.1 Optional: condition on defaults (`not.fully.paid = 1`)

We can ask CTGAN to specifically generate records where the loan is **not fully paid**, which is useful for oversampling the rare default class.


In [14]:
# Generate a larger number of synthetic samples with a strong bias towards the condition

large_synthetic_sample = ctgan.sample(5000)

# Filter to keep only the records where 'not.fully.paid' is 1
# Then sample 2000 from this filtered set. Using replace=True to handle cases where fewer than 2000 are initially generated.
synthetic_defaults = large_synthetic_sample[large_synthetic_sample[target_col] == 1].sample(n=2000, random_state=42, replace=True)

synthetic_defaults[target_col].value_counts(normalize=True)

,proportion
not.fully.paid,
1,1.0


## 6. Is the Synthetic Data Trustworthy?

We will evaluate the generated synthetic data set along the three pillars of quality.

1. **Fidelity**: Does the synthetic data statistically resemble the real data?
2. **Utility**: Can a machine learning algorithm trained on synthetic data perform well on real data?
3. **Privacy**: Can we guarantee that the synthetic data does not expose sentitive information from the real data?

### 6.1 Fidelity: do synthetic and real loans look statistically similar?

We evaluate fidelity for numeric columns based on individual columns and pairwise correlations

#### 6.1.1 Univariate Fidelity:

- For each numeric column, run a **Kolmogorov–Smirnov (KS) test** comparing real vs synthetic samples.
- KS statistic near 0 ⇒ distributions are similar.
- Higher KS ⇒ synthetic deviates from real.

We do this feature by feature.


In [15]:
real = ctgan_data
syn = synthetic

ks_results = {}

for col in num_cols:
    r = real[col].sample(min(5000, len(real)), random_state=42)
    s = syn[col].sample(min(5000, len(syn)), random_state=42)
    stat, pval = ks_2samp(r, s)
    ks_results[col] = {"ks_stat": stat, "p_value": pval}

ks_df = pd.DataFrame(ks_results).T.sort_values("ks_stat")
ks_df


,ks_stat,p_value
days.with.cr.line,0.0406,5.260608e-04
pub.rec,0.0626,6.121204e-09
revol.util,0.0676,2.351501e-10
dti,0.0864,1.181610e-16
revol.bal,0.1070,2.481331e-25
installment,0.1170,3.239998e-30
fico,0.1196,1.474321e-31
log.annual.inc,0.1438,1.765460e-45
int.rate,0.1570,3.628612e-54
delinq.2yrs,0.1902,1.875564e-79


Interpretation:

- Which features have the **lowest** KS (best‑matched distributions)?
- Which features are hardest for CTGAN to mimic (highest KS)?



###' 6.1.2 Correlation Structure

We now compare **pairwise correlations** between numerical features in real vs synthetic data.

- Compute correlation matrices for real and synthetic numeric features.
- Look at absolute differences between them.


In [16]:
real_corr = real[num_cols].corr()
syn_corr = syn[num_cols].corr()

corr_diff = (real_corr - syn_corr).abs()
corr_diff


,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec
int.rate,0.000000,0.105027,0.007613,0.057948,0.220844,0.040503,0.079040,0.029412,0.003392,0.034409,0.027526
installment,0.105027,0.000000,0.155209,0.205976,0.161908,0.069901,0.013848,0.227788,0.024605,0.036467,0.029314
log.annual.inc,0.007613,0.155209,0.000000,0.252382,0.088081,0.151675,0.172380,0.148596,0.066656,0.092209,0.028693
dti,0.057948,0.205976,0.252382,0.000000,0.016590,0.028767,0.051491,0.067534,0.012951,0.021235,0.033757
fico,0.220844,0.161908,0.088081,0.016590,0.000000,0.059869,0.085179,0.133876,0.119035,0.039434,0.003666
days.with.cr.line,0.040503,0.069901,0.151675,0.028767,0.059869,0.000000,0.204194,0.014969,0.152927,0.107809,0.094528
revol.bal,0.079040,0.013848,0.172380,0.051491,0.085179,0.204194,0.000000,0.078492,0.018166,0.028987,0.005072
revol.util,0.029412,0.227788,0.148596,0.067534,0.133876,0.014969,0.078492,0.000000,0.138831,0.089527,0.003087
inq.last.6mths,0.003392,0.024605,0.066656,0.012951,0.119035,0.152927,0.018166,0.138831,0.000000,0.013601,0.056778
delinq.2yrs,0.034409,0.036467,0.092209,0.021235,0.039434,0.107809,0.028987,0.089527,0.013601,0.000000,0.057878


In [17]:
# Average absolute difference in correlation per feature
corr_diff.mean().sort_values(ascending=False)


,0
log.annual.inc,0.105772
installment,0.093640
revol.util,0.084737
fico,0.084407
days.with.cr.line,0.084104
dti,0.068057
revol.bal,0.066986
inq.last.6mths,0.055176
int.rate,0.055065
delinq.2yrs,0.047414


### 6.2 Utility: can synthetic data train a useful default model?

We evaluate **utility** using the TSTR protocol:

1. Fit a classifier on **synthetic** data.
2. Test on **held‑out real** data.
3. Compare performance with a classifier trained on **real** data (upper bound).

Metrics:

- ROC AUC
- F1‑score (for imbalanced classification)


#### 6.2.1 Create encoders/scaler on REAL training data

We will:

- Fit **OneHotEncoder** and **StandardScaler** only on the **real training** subset.
- Apply exactly the same transformations to synthetic data and real test data.


In [18]:
# Fit on REAL TRAINING DATA ONLY
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

ohe.fit(real_train[cat_cols])
scaler.fit(real_train[num_cols])

def preprocess_for_model(df_subset):
    X_cat = df_subset[cat_cols]
    X_num = df_subset[num_cols]
    y_out = df_subset[target_col].astype(int)

    X_cat_enc = ohe.transform(X_cat)
    X_num_scaled = scaler.transform(X_num)

    X_all = np.hstack([X_cat_enc, X_num_scaled])
    return X_all, y_out


#### 6.2.2 Baseline: Train on REAL, Test on REAL

This is our **upper bound** for performance.


In [19]:
X_real_train, y_real_train = preprocess_for_model(real_train)
X_real_test, y_real_test = preprocess_for_model(real_test)

rf_real = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_real.fit(X_real_train, y_real_train)

y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real = f1_score(y_real_test, y_pred_real)

auc_real, f1_real


(np.float64(0.6691897976164206), 0.0375)

#### 6.2.3 TSTR: Train on SYNTHETIC, Test on REAL

We now:

- Use our `synthetic` dataframe as training data.
- Evaluate on the **same real test set** as above.

This tells us how good models trained purely on synthetic data are for predicting `not.fully.paid` on real loans.


In [20]:
X_syn_train, y_syn_train = preprocess_for_model(synthetic)

rf_syn = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_syn.fit(X_syn_train, y_syn_train)

y_proba_syn = rf_syn.predict_proba(X_real_test)[:, 1]
y_pred_syn = (y_proba_syn >= 0.5).astype(int)

auc_syn = roc_auc_score(y_real_test, y_proba_syn)
f1_syn = f1_score(y_real_test, y_pred_syn)

auc_syn, f1_syn


(np.float64(0.6318610908104453), 0.1620253164556962)

#### 6.2.4 Compare utility: TRTR vs TSTR


In [21]:
pd.DataFrame(
    {
        "AUC": [auc_real, auc_syn],
        "F1": [f1_real, f1_syn]
    },
    index=["Train REAL, Test REAL", "Train SYNTHETIC, Test REAL"]
)


,AUC,F1
"Train REAL, Test REAL",0.669190,0.037500
"Train SYNTHETIC, Test REAL",0.631861,0.162025


### 6.3 Privacy: Approximate Memorization Check

CTGAN can overfit and memorize real rows, which is a privacy risk.

A simple heuristic:
- Take numeric features from synthetic data.
- For each synthetic point, compute the distance to the **nearest real point**.
- If many synthetic points are at extremely small distance, it may indicate memorization.

This is not a formal privacy guarantee, but a useful metric.


In [22]:
# Sample down for speed
real_num = real[num_cols].sample(5000, random_state=42)
syn_num = syn[num_cols].sample(5000, random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

distances, indices = nn.kneighbors(syn_num)
distances = distances.flatten()

pd.Series(distances).describe()


,0
count,5000.000000
mean,387.965182
std,559.097899
min,24.704910
25%,134.636100
50%,207.996480
75%,379.508078
max,7803.285156


Interpretation:

- Very small distances (e.g., many < 1e-6 after scaling) might indicate memorization.
- Larger distances suggest synthetic records are not exact copies.

For a production‑grade system, you would combine such checks with more formal privacy metrics (e.g., membership inference tests, differential privacy variants of CTGAN).


## 7. What did we learn?

In this notebook we:

1. Treated **`not.fully.paid` as the key outcome** for loan default risk.
2. Trained **CTGAN** on mixed‑type loan data to generate synthetic loans.
3. Evaluated **fidelity**:
   - KS test per numeric feature.
   - Correlation structure differences.
4. Evaluated **utility** via **Train‑Synthetic‑Test‑Real (TSTR)** against a real‑trained baseline.
5. Ran a simple **privacy heuristic** using nearest‑neighbor distances.



#Part 2 -  TVAE: Tabular Variational Autoencoder

We now train **TVAE**, another SDV model for tabular data.  
TVAE models the joint distribution using a variational autoencoder instead of an adversarial game.

We will:

1. Train TVAE on the same columns as CTGAN.
2. Generate synthetic loans.
3. Reuse the same evaluation pipeline (fidelity + TSTR utility).


In [23]:
from sdv.single_table import TVAESynthesizer

In [24]:
from sdv.metadata import SingleTableMetadata

tvae_data = df[cat_cols + num_cols + [target_col]].copy()

discrete_cols = cat_cols + ["inq.last.6mths", "delinq.2yrs", "pub.rec", target_col]

# Create metadata from the training data
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(tvae_data)

# Explicitly set discrete columns in the metadata
for col in discrete_cols:
    metadata.update_column(column_name=col, sdtype='categorical')

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=300,         # similar budget to CTGAN for fairness
    batch_size=500
)

tvae.fit(tvae_data)

### Generate synthetic loans with TVAE

We generate the same number of records (10,000) to make comparisons fair.


In [25]:
n_synth = 10_000
synthetic_tvae = tvae.sample(n_synth)

synthetic_tvae.head()


,purpose,credit.policy,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,debt_consolidation,1,0.1242,487.41,11.320181,11.24,734,5711.350998,31998,47.16,0,0,0,0
1,debt_consolidation,1,0.0778,157.53,11.051193,7.79,812,6363.967847,7413,24.57,0,0,0,0
2,debt_consolidation,1,0.1528,256.04,10.912236,20.66,673,3309.436105,10216,80.02,0,0,0,0
3,debt_consolidation,1,0.1105,528.96,11.720089,5.43,753,7232.206012,31531,36.29,0,0,0,0
4,all_other,1,0.0735,101.20,10.732216,1.30,782,4343.807874,2176,8.03,0,0,0,0


In [26]:
print("TVAE synthetic outcome distribution:")
print(synthetic_tvae[target_col].value_counts(normalize=True))


TVAE synthetic outcome distribution:
not.fully.paid
0    0.9999
1    0.0001
Name: proportion, dtype: float64


## Fidelity: CTGAN vs TVAE (KS test)

We compute the KS statistic per numeric feature for:

- Real vs **CTGAN** synthetic
- Real vs **TVAE** synthetic

Lower KS ⇒ closer match to real marginal distribution. [web:42][web:140]


In [27]:
real = df[cat_cols + num_cols + [target_col]].copy()

def ks_per_feature(real_df, syn_df, num_cols):
    results = {}
    for col in num_cols:
        r = real_df[col].sample(min(5000, len(real_df)), random_state=42)
        s = syn_df[col].sample(min(5000, len(syn_df)), random_state=42)
        stat, pval = ks_2samp(r, s)
        results[col] = {"ks_stat": stat, "p_value": pval}
    return pd.DataFrame(results).T

ks_ctgan = ks_per_feature(real, synthetic, num_cols)
ks_tvae  = ks_per_feature(real, synthetic_tvae, num_cols)

ks_compare = pd.DataFrame({
    "KS_CTGAN": ks_ctgan["ks_stat"],
    "KS_TVAE": ks_tvae["ks_stat"]
}).sort_values("KS_CTGAN")

ks_compare


,KS_CTGAN,KS_TVAE
days.with.cr.line,0.0406,0.1426
pub.rec,0.0626,0.0546
revol.util,0.0676,0.0814
dti,0.0864,0.1244
revol.bal,0.1070,0.0494
installment,0.1170,0.0838
fico,0.1196,0.0458
log.annual.inc,0.1438,0.0764
int.rate,0.1570,0.0630
delinq.2yrs,0.1902,0.1176


You can quickly see:

- Which model better fits each numeric feature.
- Whether one model tends to systematically have lower KS across features.


### Correlation structure: CTGAN vs TVAE

We compare correlation matrices as before, now for both synthesizers. [web:143]


In [28]:
real_corr = real[num_cols].corr()
ctgan_corr = synthetic[num_cols].corr()
tvae_corr  = synthetic_tvae[num_cols].corr()

ctgan_corr_diff = (real_corr - ctgan_corr).abs()
tvae_corr_diff  = (real_corr - tvae_corr).abs()

corr_compare = pd.DataFrame({
    "mean_abs_diff_CTGAN": ctgan_corr_diff.mean(),
    "mean_abs_diff_TVAE": tvae_corr_diff.mean()
}).sort_values("mean_abs_diff_CTGAN")

corr_compare


,mean_abs_diff_CTGAN,mean_abs_diff_TVAE
pub.rec,0.030936,NaN
delinq.2yrs,0.047414,0.049408
int.rate,0.055065,0.072914
inq.last.6mths,0.055176,0.029743
revol.bal,0.066986,0.071231
dti,0.068057,0.097777
days.with.cr.line,0.084104,0.129299
fico,0.084407,0.083328
revol.util,0.084737,0.100721
installment,0.093640,0.123791


## Utility: TSTR for CTGAN vs TVAE

We reuse the same **Train‑Synthetic‑Test‑Real** pipeline: [web:39][web:148]

- Encoders (`ohe`) and `scaler` were fit on **real_train**.
- `preprocess_for_model` converts any dataframe to model‑ready `X`, `y`.


In [29]:
X_tvae_train, y_tvae_train = preprocess_for_model(synthetic_tvae)

rf_tvae = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_tvae.fit(X_tvae_train, y_tvae_train)

y_proba_tvae = rf_tvae.predict_proba(X_real_test)[:, 1]
y_pred_tvae = (y_proba_tvae >= 0.5).astype(int)

auc_tvae = roc_auc_score(y_real_test, y_proba_tvae)
f1_tvae = f1_score(y_real_test, y_pred_tvae)

auc_tvae, f1_tvae


(np.float64(0.5155902365156905), 0.0)

### Compare TRTR, CTGAN‑TSTR, TVAE‑TSTR


In [30]:
utility_df = pd.DataFrame(
    {
        "AUC": [auc_real, auc_syn, auc_tvae],
        "F1":  [f1_real,  f1_syn,  f1_tvae]
    },
    index=[
        "Train REAL, Test REAL",
        "Train CTGAN, Test REAL",
        "Train TVAE, Test REAL"
    ]
)

utility_df


,AUC,F1
"Train REAL, Test REAL",0.669190,0.037500
"Train CTGAN, Test REAL",0.631861,0.162025
"Train TVAE, Test REAL",0.515590,0.000000


Discussion prompts:

- Which synthesizer gives a classifier whose AUC/F1 is closer to the **real‑trained** baseline?
- Are there differences in calibration or class balance that might explain performance? [web:148][web:143]


## Privacy heuristic: nearest‑neighbor distance (CTGAN vs TVAE)

We reuse the **nearest‑neighbor distance** approach to compare memorization risk for the two models. [web:140][web:146]


In [31]:
# Sample real numeric subset for reference
real_num = real[num_cols].sample(5000, random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

# CTGAN
syn_ctgan_num = synthetic[num_cols].sample(5000, random_state=42)
dist_ctgan, _ = nn.kneighbors(syn_ctgan_num)
dist_ctgan = dist_ctgan.flatten()

# TVAE
syn_tvae_num = synthetic_tvae[num_cols].sample(5000, random_state=42)
dist_tvae, _ = nn.kneighbors(syn_tvae_num)
dist_tvae = dist_tvae.flatten()

privacy_df = pd.DataFrame(
    {
        "CTGAN_dist": dist_ctgan,
        "TVAE_dist": dist_tvae
    }
)

privacy_df.describe()


,CTGAN_dist,TVAE_dist
count,5000.000000,5000.000000
mean,387.965182,224.723519
std,559.097899,317.015493
min,24.704910,14.190370
25%,134.636100,112.690665
50%,207.996480,160.473432
75%,379.508078,239.434867
max,7803.285156,7124.952023


Interpretation:

- Higher typical nearest‑neighbor distances suggest less memorization.
- If one model has consistently much smaller distances, it may be overfitting more to specific real points. [web:140][web:146]


## CTGAN vs TVAE: what to observe


1. **Fidelity (KS & correlation):**
   - For which features does CTGAN best mimic the real distribution?
   - Where does TVAE do better?
   - Is one model more consistent across features?

2. **Utility (TSTR AUC & F1):**
   - Which synthetic dataset produces more useful classifiers for predicting `not.fully.paid`?
   - How close are they to the real‑trained baseline?

3. **Privacy heuristic (NN distances):**
   - Does either model appear to memorize real records more?

This gives a concrete, model‑agnostic way to discuss **fidelity–utility–privacy trade‑offs** across two popular tabular synthesizers on a realistic loan default problem.


Required Task 13
(check the python notebook SD4_Synthetic_Loan_Data_Generation_with_CTGAN_&_TVAE.ipynb for context and further details)

Load the file fraud_transactions.csv and create a synthetic data set of 5000 records. Evaluate the quality of the synthetic data created.


In [32]:
"""
Task 13 — Synthetic Fraud Transaction Data Generation & Evaluation
==================================================================
Standalone script. No Colab / notebook dependency.

Requirements (install once):
    pip install sdv ctgan pandas numpy scikit-learn scipy matplotlib seaborn

Usage:
    python task13_synthetic_fraud.py

Make sure fraud_transactions.csv is in the SAME folder as this script.
"""

# ==============================================================================
# 0.  CONFIG  — edit only this block if needed
# ==============================================================================
CSV_PATH        = "/content/fraud_transactions.csv"   # path to your input file
N_SYNTH         = 5000                       # synthetic records to generate
EPOCHS          = 300                        # training epochs (lower = faster but worse quality)
BATCH_SIZE      = 500
RANDOM_SEED     = 42

# ==============================================================================
# 1.  IMPORTS
# ==============================================================================
import os, sys, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.neighbors import NearestNeighbors

# ==============================================================================
# 2.  LOAD & INSPECT DATA
# ==============================================================================
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        sys.exit(f"\n❌  File not found: '{path}'\n"
                 f"    Place fraud_transactions.csv in the same folder as this script.\n")

    df = pd.read_csv(path)
    print("=" * 65)
    print("  Task 13 — Synthetic Fraud Data Generation & Evaluation")
    print("=" * 65)
    print(f"\n📂  Loaded '{path}'  →  {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("\n── Column overview ──────────────────────────────────────────")
    print(df.dtypes.to_string())
    print("\n── First 3 rows ─────────────────────────────────────────────")
    print(df.head(3).to_string())
    print("\n── Missing values ───────────────────────────────────────────")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.any() else "  None ✅")
    return df


# ==============================================================================
# 3.  AUTO-DETECT COLUMN TYPES
# ==============================================================================
def detect_column_types(df: pd.DataFrame):
    """
    Automatically separate columns into:
      - target_col  : a binary fraud/label column if one exists
      - cat_cols    : categorical or low-cardinality columns
      - num_cols    : numeric columns
      - drop_cols   : id/timestamp columns to exclude from synthesis
    """
    # Heuristic: drop obvious ID or timestamp columns
    drop_keywords = ["id", "trans_num", "unnamed", "index"]
    drop_cols = [c for c in df.columns
                 if any(k in c.lower() for k in drop_keywords)]

    # Try to find a binary fraud/target column
    fraud_keywords = ["fraud", "is_fraud", "label", "class", "target"]
    target_col = None
    for kw in fraud_keywords:
        for col in df.columns:
            if kw in col.lower() and df[col].nunique() == 2:
                target_col = col
                break
        if target_col:
            break

    remaining = [c for c in df.columns if c not in drop_cols]

    # Separate categoricals vs numerics
    cat_cols, num_cols = [], []
    for col in remaining:
        if col == target_col:
            continue
        if df[col].dtype == object or df[col].nunique() <= 20:
            cat_cols.append(col)
        else:
            num_cols.append(col)

    print("\n── Auto-detected column types ───────────────────────────────")
    print(f"  Target column  : {target_col}")
    print(f"  Categorical    : {cat_cols}")
    print(f"  Numeric        : {num_cols}")
    print(f"  Dropped (ID/ts): {drop_cols}")

    return target_col, cat_cols, num_cols, drop_cols


# ==============================================================================
# 4.  CTGAN SYNTHESIS
# ==============================================================================
def train_ctgan(train_df, cat_cols, num_cols, target_col, discrete_extra=None):
    from ctgan import CTGAN

    cols = cat_cols + num_cols + ([target_col] if target_col else [])
    data = train_df[cols].copy()

    discrete = list(cat_cols) + (discrete_extra or [])
    if target_col:
        discrete.append(target_col)
    # Include integer-looking numeric cols as discrete
    for c in num_cols:
        if data[c].dropna().apply(float.is_integer if hasattr(data[c].iloc[0], 'is_integer') else lambda x: x == int(x)).all():
            if c not in discrete:
                discrete.append(c)

    print("\n🤖  Training CTGAN …")
    model = CTGAN(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True)
    model.fit(data, discrete_columns=discrete)
    return model, cols, discrete


def train_tvae(train_df, cat_cols, num_cols, target_col, discrete_extra=None):
    from sdv.single_table import TVAESynthesizer
    from sdv.metadata import SingleTableMetadata

    cols = cat_cols + num_cols + ([target_col] if target_col else [])
    data = train_df[cols].copy()

    discrete = list(cat_cols) + (discrete_extra or [])
    if target_col:
        discrete.append(target_col)
    for c in num_cols:
        try:
            if data[c].dropna().apply(lambda x: x == int(x)).all():
                if c not in discrete:
                    discrete.append(c)
        except Exception:
            pass

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(data)
    for col in discrete:
        if col in data.columns:
            metadata.update_column(column_name=col, sdtype='categorical')

    print("\n🤖  Training TVAE …")
    model = TVAESynthesizer(metadata=metadata, epochs=EPOCHS, batch_size=BATCH_SIZE)
    model.fit(data)
    return model, cols


# ==============================================================================
# 5.  EVALUATION HELPERS
# ==============================================================================

# ── 5a. Fidelity — KS test ──────────────────────────────────────────────────
def ks_per_feature(real_df, syn_df, num_cols, n=5000):
    results = {}
    for col in num_cols:
        r = real_df[col].dropna().sample(min(n, len(real_df)), random_state=RANDOM_SEED)
        s = syn_df[col].dropna().sample(min(n, len(syn_df)), random_state=RANDOM_SEED)
        stat, pval = ks_2samp(r, s)
        results[col] = {"ks_stat": round(stat, 4), "p_value": round(pval, 4)}
    return pd.DataFrame(results).T.sort_values("ks_stat")


# ── 5b. Fidelity — Correlation difference ───────────────────────────────────
def correlation_diff(real_df, syn_df, num_cols):
    real_corr = real_df[num_cols].corr()
    syn_corr  = syn_df[num_cols].corr()
    diff      = (real_corr - syn_corr).abs()
    return diff.mean().sort_values(ascending=False)


# ── 5c. Utility — TSTR ──────────────────────────────────────────────────────
def build_preprocessor(real_train, cat_cols, num_cols, target_col):
    ohe    = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    scaler = StandardScaler()

    if cat_cols:
        ohe.fit(real_train[cat_cols])
    if num_cols:
        scaler.fit(real_train[num_cols])

    def preprocess(df_sub):
        parts = []
        if cat_cols:
            parts.append(ohe.transform(df_sub[cat_cols]))
        if num_cols:
            parts.append(scaler.transform(df_sub[num_cols]))
        X = np.hstack(parts) if parts else np.zeros((len(df_sub), 1))
        y = df_sub[target_col].astype(int)
        return X, y

    return preprocess


def tstr_eval(X_train, y_train, X_test, y_test, label=""):
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_proba = rf.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_proba)
    f1  = f1_score(y_test, y_pred, zero_division=0)
    print(f"  {label:<35} AUC={auc:.4f}  F1={f1:.4f}")
    return auc, f1


# ── 5d. Privacy — Nearest-neighbour distances ────────────────────────────────
def nn_privacy(real_df, syn_df, num_cols, n=3000):
    real_num = real_df[num_cols].dropna().sample(min(n, len(real_df)), random_state=RANDOM_SEED)
    syn_num  = syn_df[num_cols].dropna().sample(min(n, len(syn_df)), random_state=RANDOM_SEED)
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(real_num)
    dist, _ = nn.kneighbors(syn_num)
    return dist.flatten()


# ==============================================================================
# 6.  MAIN
# ==============================================================================
def main():
    # ── 6a. Load data ─────────────────────────────────────────────────────────
    df = load_data(CSV_PATH)
    df = df.dropna().reset_index(drop=True)

    target_col, cat_cols, num_cols, drop_cols = detect_column_types(df)

    if not target_col:
        print("\n⚠️  No binary target column detected. Utility (TSTR) evaluation will be skipped.")

    all_cols = cat_cols + num_cols + ([target_col] if target_col else [])
    df_model = df[all_cols].copy()

    # ── 6b. Train / test split ────────────────────────────────────────────────
    if target_col:
        real_train, real_test = train_test_split(
            df_model, test_size=0.2, random_state=RANDOM_SEED,
            stratify=df_model[target_col]
        )
    else:
        real_train, real_test = train_test_split(
            df_model, test_size=0.2, random_state=RANDOM_SEED
        )
    print(f"\n✂️   Train/test split → train={len(real_train):,}  test={len(real_test):,}")

    # ── 6c. Train CTGAN & generate ───────────────────────────────────────────
    ctgan_model, ctgan_cols, _ = train_ctgan(real_train, cat_cols, num_cols, target_col)
    synthetic_ctgan = ctgan_model.sample(N_SYNTH)
    print(f"\n✅  CTGAN generated {len(synthetic_ctgan):,} synthetic records.")

    if target_col:
        print("\n  Class distribution (CTGAN synthetic):")
        print(synthetic_ctgan[target_col].value_counts(normalize=True).to_string())

    # ── 6d. Train TVAE & generate ─────────────────────────────────────────────
    tvae_model, tvae_cols = train_tvae(real_train, cat_cols, num_cols, target_col)
    synthetic_tvae = tvae_model.sample(N_SYNTH)
    print(f"\n✅  TVAE generated {len(synthetic_tvae):,} synthetic records.")

    if target_col:
        print("\n  Class distribution (TVAE synthetic):")
        print(synthetic_tvae[target_col].value_counts(normalize=True).to_string())

    # ==========================================================================
    # 7.  EVALUATION
    # ==========================================================================
    print("\n" + "=" * 65)
    print("  EVALUATION")
    print("=" * 65)

    # ── 7a. Fidelity — KS test ────────────────────────────────────────────────
    if num_cols:
        print("\n📊  [FIDELITY] Kolmogorov-Smirnov test (lower = better match)\n")
        ks_ctgan = ks_per_feature(df_model, synthetic_ctgan, num_cols)
        ks_tvae  = ks_per_feature(df_model, synthetic_tvae,  num_cols)

        ks_compare = pd.DataFrame({
            "KS_CTGAN": ks_ctgan["ks_stat"],
            "KS_TVAE":  ks_tvae["ks_stat"]
        }).sort_values("KS_CTGAN")

        print(ks_compare.to_string())
        print(f"\n  Mean KS → CTGAN: {ks_compare['KS_CTGAN'].mean():.4f}  "
              f"|  TVAE: {ks_compare['KS_TVAE'].mean():.4f}")

        # ── 7b. Fidelity — Correlation ────────────────────────────────────────
        if len(num_cols) > 1:
            print("\n📊  [FIDELITY] Mean absolute correlation difference (lower = better)\n")
            corr_ctgan = correlation_diff(df_model, synthetic_ctgan, num_cols)
            corr_tvae  = correlation_diff(df_model, synthetic_tvae,  num_cols)
            corr_compare = pd.DataFrame({
                "Corr_diff_CTGAN": corr_ctgan,
                "Corr_diff_TVAE":  corr_tvae
            }).sort_values("Corr_diff_CTGAN")
            print(corr_compare.to_string())
            print(f"\n  Overall mean → CTGAN: {corr_ctgan.mean():.4f}  "
                  f"|  TVAE: {corr_tvae.mean():.4f}")
    else:
        print("\n⚠️  No numeric columns detected — skipping KS and correlation tests.")

    # ── 7c. Utility — TSTR ────────────────────────────────────────────────────
    if target_col and num_cols:
        print("\n📊  [UTILITY] Train-Synthetic-Test-Real (TSTR)\n")
        preprocess = build_preprocessor(real_train, cat_cols, num_cols, target_col)

        X_real_train, y_real_train = preprocess(real_train)
        X_real_test,  y_real_test  = preprocess(real_test)
        X_ctgan,      y_ctgan      = preprocess(synthetic_ctgan)
        X_tvae,       y_tvae       = preprocess(synthetic_tvae)

        auc_real, f1_real = tstr_eval(X_real_train, y_real_train,
                                       X_real_test,  y_real_test,
                                       "Train REAL → Test REAL (baseline)")
        auc_ctgan, f1_ctgan = tstr_eval(X_ctgan, y_ctgan,
                                         X_real_test, y_real_test,
                                         "Train CTGAN → Test REAL")
        auc_tvae, f1_tvae   = tstr_eval(X_tvae, y_tvae,
                                         X_real_test, y_real_test,
                                         "Train TVAE → Test REAL")

        utility_df = pd.DataFrame({
            "AUC": [auc_real, auc_ctgan, auc_tvae],
            "F1":  [f1_real,  f1_ctgan,  f1_tvae]
        }, index=["TRTR (baseline)", "CTGAN TSTR", "TVAE TSTR"])

        print(f"\n{utility_df.to_string()}")
        print(f"\n  CTGAN utility gap → AUC: {auc_real - auc_ctgan:+.4f}  "
              f"F1: {f1_real - f1_ctgan:+.4f}")
        print(f"  TVAE  utility gap → AUC: {auc_real - auc_tvae:+.4f}  "
              f"F1: {f1_real - f1_tvae:+.4f}")
    else:
        print("\n⚠️  Skipping TSTR (no target column or no numeric features).")

    # ── 7d. Privacy — NN distances ────────────────────────────────────────────
    if num_cols:
        print("\n📊  [PRIVACY] Nearest-neighbour distance heuristic\n")
        print("  (Higher distance = less memorisation risk)\n")
        dist_ctgan = nn_privacy(df_model, synthetic_ctgan, num_cols)
        dist_tvae  = nn_privacy(df_model, synthetic_tvae,  num_cols)

        privacy_df = pd.DataFrame({
            "CTGAN_dist": dist_ctgan,
            "TVAE_dist":  dist_tvae
        })
        print(privacy_df.describe().round(4).to_string())

        ctgan_zero = (dist_ctgan < 1e-4).mean() * 100
        tvae_zero  = (dist_tvae  < 1e-4).mean() * 100
        print(f"\n  Near-exact copies (<1e-4 dist) → "
              f"CTGAN: {ctgan_zero:.1f}%  |  TVAE: {tvae_zero:.1f}%")

    # ==========================================================================
    # 8.  SAVE OUTPUTS
    # ==========================================================================
    ctgan_out = "synthetic_fraud_ctgan.csv"
    tvae_out  = "synthetic_fraud_tvae.csv"
    synthetic_ctgan.to_csv(ctgan_out, index=False)
    synthetic_tvae.to_csv(tvae_out,  index=False)

    print("\n" + "=" * 65)
    print("  SUMMARY")
    print("=" * 65)
    print(f"  Real data size       : {len(df_model):,} records")
    print(f"  Synthetic (CTGAN)    : {len(synthetic_ctgan):,} records  → saved to '{ctgan_out}'")
    print(f"  Synthetic (TVAE)     : {len(synthetic_tvae):,} records  → saved to '{tvae_out}'")
    if num_cols:
        print(f"\n  Best fidelity model  : {'CTGAN' if ks_compare['KS_CTGAN'].mean() < ks_compare['KS_TVAE'].mean() else 'TVAE'} "
              f"(lower mean KS)")
    if target_col and num_cols:
        best_util = "CTGAN" if auc_ctgan >= auc_tvae else "TVAE"
        print(f"  Best utility model   : {best_util} (higher TSTR AUC)")
    if num_cols:
        best_priv = "CTGAN" if dist_ctgan.mean() >= dist_tvae.mean() else "TVAE"
        print(f"  Best privacy model   : {best_priv} (higher NN distance)")
    print("=" * 65 + "\n")


if __name__ == "__main__":
    main()

  Task 13 — Synthetic Fraud Data Generation & Evaluation

📂  Loaded '/content/fraud_transactions.csv'  →  6,486 rows × 8 columns

── Column overview ──────────────────────────────────────────
trans_date_trans_time     object
merchant                  object
category                  object
amt                      float64
gender                    object
state                     object
job                       object
is_fraud                   int64

── First 3 rows ─────────────────────────────────────────────
  trans_date_trans_time                             merchant       category    amt gender state                      job  is_fraud
0         2/27/19 21:32  fraud_Langosh, Wintheiser and Hyatt    food_dining  83.64      F    TX       Physicist, medical         0
1         2/13/19 19:41               fraud_Dibbert and Sons  entertainment  79.13      M    PA  Secretary/administrator         0
2         1/11/19 20:03   fraud_McDermott, Osinski and Morar           home  12.02      

Gen. (+00.18) | Discrim. (+00.01): 100%|██████████| 300/300 [10:18<00:00,  2.06s/it]



✅  CTGAN generated 5,000 synthetic records.

  Class distribution (CTGAN synthetic):
is_fraud
0    0.799
1    0.201

🤖  Training TVAE …

✅  TVAE generated 5,000 synthetic records.

  Class distribution (TVAE synthetic):
is_fraud
0    0.9302
1    0.0698

  EVALUATION

📊  [FIDELITY] Kolmogorov-Smirnov test (lower = better match)

     KS_CTGAN  KS_TVAE
amt    0.1356   0.0814

  Mean KS → CTGAN: 0.1356  |  TVAE: 0.0814

📊  [UTILITY] Train-Synthetic-Test-Real (TSTR)

  Train REAL → Test REAL (baseline)   AUC=0.9737  F1=0.8276
  Train CTGAN → Test REAL             AUC=0.7491  F1=0.5793
  Train TVAE → Test REAL              AUC=0.9511  F1=0.3276

                      AUC        F1
TRTR (baseline)  0.973652  0.827586
CTGAN TSTR       0.749105  0.579310
TVAE TSTR        0.951085  0.327586

  CTGAN utility gap → AUC: +0.2245  F1: +0.2483
  TVAE  utility gap → AUC: +0.0226  F1: +0.5000

📊  [PRIVACY] Nearest-neighbour distance heuristic

  (Higher distance = less memorisation risk)

       CTGA